---
## 1. Import Libraries

## Dataset Cleaning: AIHW Aged Care Service List
### Purpose
This notebook cleans and prepares the AIHW Aged Care Service List for use in the **Epic 3 – Find Local Digital Help** feature.  
The goal of Epic 3 is to help Australians aged 60+ find nearby physical aged care services and community programs where they can get in-person support.


In [2]:
import pandas as pd
import os

# show all columns and reasonable row width
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

---
## 2. Load the Raw Dataset

The Excel file has **2 metadata rows** at the top before the actual column headers.  
We use `skiprows=2` to skip them and treat row 3 as the header.

In [34]:
# Move from src/notebooks → src
src_base_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
data_path = os.path.join(src_base_path, 'data', 'raw_data')
clean_data_path = os.path.join(src_base_path, 'data', 'clean_data')

# Create clean_data folder if it doesn't exist
os.makedirs(clean_data_path, exist_ok=True)

RAW_PATH = os.path.join(data_path, 'Service-List-2025-Australia.xlsx')
CLEAN_PATH = os.path.join(clean_data_path, 'cleaned_AIHW_ServiceList.csv')

print("Raw data path :", data_path)
print("Clean data path:", clean_data_path)
print("File exists   :", os.path.exists(RAW_PATH))

# Load raw data , skip the 2 title rows at the top of the Excel sheet
df_raw = pd.read_excel(RAW_PATH, sheet_name=0, skiprows=2, header=0)

print(f'\nRaw dataset loaded successfully.')
print(f'Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')

Raw data path : C:\Monash\SEM4\FIT-5120-main\FIT-5120-Main-project\src\data\raw_data
Clean data path: C:\Monash\SEM4\FIT-5120-main\FIT-5120-Main-project\src\data\clean_data
File exists   : True

Raw dataset loaded successfully.
Shape: 5378 rows x 25 columns


---
## 3. Exploratory Data Analysis (EDA)

Before any cleaning, we explore the dataset to understand its structure,  
data types, missing values, and key distributions.  


### 3.1 First Look, Sample Rows

In [8]:
# Preview the first 5 rows to understand what the data looks like
df_raw.head()

,Service Name,Physical Address,Physical Suburb,Physical State,Physical Post Code,2018 Aged Care Planning Region (ACPR),Care Type,Residential Places,Home Care Places,Restorative Care Places,Provider Name,Organisation Type,ABS Remoteness,2019 MMM Code,2016 SA2 Code,2016 SA2 Name,2016 SA3 Code,2016 SA3 Name,2023 LGA Name,2023 LGA Code,2017 PHN Code,2017 PHN Name,Latitude,Longitude,2024-25 Australian Government Funding
0,Braidwood Multi-Purpose Service,77 Monkittee Street,BRAIDWOOD,NSW,2622,Southern Highlands,Multi-Purpose Service,37,2,,Southern NSW Local Health District,State Government,Inner Regional Australia,5,11007,Braidwood,10102,Queanbeyan,Queanbeyan-Palerang,16490,PHN106,South Eastern NSW,-35.442764,149.805407,2683564.28
1,Dellacourt,42 NICHOLSON Place,WEST ALBURY,NSW,2640,Riverina/Murray,Residential,122,,,Lutheran Aged Care Albury,Charitable,Inner Regional Australia,2,11174,Albury - South,10901,Albury,Albury,10050,PHN205,Murray,-36.075401,146.890896,13628406.71
2,Brerrina Multi-Purpose Service,56 DOYLE Street,BRERRINA,NSW,2839,Orana Far West,Multi-Purpose Service,12,4,,Western NSW Local Health District,State Government,Very Remote Australia,7,11092,Bourke - Brewarrina,10501,Bourke - Cobar - Coonamble,Brewarrina,11200,PHN107,Western NSW,-29.961591,146.864313,1372056.96
3,BaptistCare Maranoa Centre - Alstonville,15 The Avenue -,ALSTONVILLE,NSW,2477,Far North Coast,Residential,90,,,BaptistCare NSW & ACT,Charitable,Inner Regional Australia,3,11237,Ballina Region,11201,Richmond Valley - Coastal,Ballina,10250,PHN109,North Coast,-28.840536,153.436634,10456454.14
4,Urana Multi-Purpose Service,127-129 Princess Street,URANA,NSW,2645,Riverina/Murray,Multi-Purpose Service,19,,,Murrumbidgee Local Health District,State Government,Outer Regional Australia,5,11181,Corowa Region,10903,Upper Murray exc. Albury,Federation,12870,PHN110,Murrumbidgee,-35.325391,146.267697,1591105.21


### 3.2 Column Names and Data Types

In [9]:
# Check all column names and their inferred data types
print('Column Names and Data Types:')
print('-' * 45)
df_raw.dtypes

Column Names and Data Types:
---------------------------------------------


Service Name                              object
Physical Address                          object
Physical Suburb                           object
Physical State                            object
Physical Post Code                         int64
2018 Aged Care Planning Region (ACPR)     object
Care Type                                 object
Residential Places                         int64
Home Care Places                          object
Restorative Care Places                   object
Provider Name                             object
Organisation Type                         object
ABS Remoteness                            object
2019 MMM Code                              int64
2016 SA2 Code                              int64
2016 SA2 Name                             object
2016 SA3 Code                              int64
2016 SA3 Name                             object
2023 LGA Name                             object
2023 LGA Code                              int64
2017 PHN Code       

### 3.3 Dataset Shape and Basic Statistics

In [10]:
print(f'Total Rows    : {df_raw.shape[0]}')
print(f'Total Columns : {df_raw.shape[1]}')
print()

# Basic statistics for numeric columns
df_raw.describe()

Total Rows    : 5378
Total Columns : 25



,Physical Post Code,Residential Places,2019 MMM Code,2016 SA2 Code,2016 SA3 Code,2023 LGA Code,Latitude,Longitude,2024-25 Australian Government Funding
count,5378.000000,5378.000000,5378.000000,5378.000000,5378.000000,5378.000000,5378.000000,5378.000000,5.378000e+03
mean,3713.481592,42.443102,2.065824,27090.198959,26842.553923,30126.707698,-32.968975,144.589084,6.421556e+06
std,1420.111641,52.195255,1.628776,16163.186228,15893.863435,16506.246293,5.166509,10.478005,8.658527e+06
min,800.000000,0.000000,1.000000,11007.000000,10102.000000,10050.000000,-43.314254,113.661026,-9.243965e+05
25%,2540.000000,0.000000,1.000000,11481.250000,12202.000000,17150.000000,-37.096713,144.403048,1.281023e+06
50%,3222.000000,14.000000,1.000000,21336.000000,21205.000000,24970.000000,-33.854716,147.091677,5.013574e+06
75%,4655.000000,80.000000,3.000000,31501.750000,31802.000000,36910.000000,-31.086477,151.178206,9.659008e+06
max,7469.000000,337.000000,7.000000,91004.000000,90104.000000,99399.000000,-10.569817,167.954814,3.400068e+08


### 3.4 Missing Values

In [11]:
# Count nulls per column and show as a percentage of total rows
null_counts = df_raw.isnull().sum()
null_pct = (null_counts / len(df_raw) * 100).round(2)

null_summary = pd.DataFrame({
    'Null Count': null_counts,
    'Null %': null_pct
})

print('Missing Value Summary:')
print('-' * 35)
print(null_summary[null_summary['Null Count'] > 0].to_string())
print()
print(f'Columns with no nulls: {(null_counts == 0).sum()} / {len(null_counts)}')

Missing Value Summary:
-----------------------------------
                  Null Count  Null %
Home Care Places           1    0.02

Columns with no nulls: 24 / 25


### 3.5 Care Type Distribution

In [12]:
# Understand the breakdown of service types in the dataset
# This informs which Care Types are relevant to Epic 3
print('Care Type Distribution:')
print('-' * 45)
print(df_raw['Care Type'].value_counts().to_string())

Care Type Distribution:
---------------------------------------------
Care Type
Residential                                                         2590
Home Care                                                           2363
Multi-Purpose Service                                                183
Short-Term Restorative Care (STRC)                                   126
Transition Care                                                       67
National Aboriginal and Torres Strait Islander Aged Care Program      47
Innovative Pool                                                        2


### 3.6 State Distribution

In [13]:
# Check coverage across Australian states and territories
print('Records by State/Territory:')
print('-' * 35)
print(df_raw['Physical State'].value_counts().to_string())

Records by State/Territory:
-----------------------------------
Physical State
NSW    1704
VIC    1415
QLD    1013
WA      541
SA      414
TAS     151
NT       71
ACT      69


### 3.7 Organisation Type Distribution

In [14]:
# Understand who operates these services
print('Organisation Type Distribution:')
print('-' * 40)
print(df_raw['Organisation Type'].value_counts().to_string())

Organisation Type Distribution:
----------------------------------------
Organisation Type
Private Incorporated Body    1573
Charitable                   1405
Religious                     933
Community Based               855
State Government              514
Local Government               84
Publicly Listed Company        10
Territory Government            4


### 3.8 Check Postcode Column

In [15]:
# Postcodes should be strings not integers to prevent arithmetic and preserve formatting
print(f"Postcode dtype  : {df_raw['Physical Post Code'].dtype}")
print(f"Sample values   : {df_raw['Physical Post Code'].head(8).tolist()}")

Postcode dtype  : int64
Sample values   : [2622, 2640, 2839, 2477, 2645, 2396, 2350, 2205]


### 3.9 Inspect 'Restorative Care Places' Column

In [17]:
# This column uses a space string ' ' instead of a proper null
# for rows where restorative care is not applicable
print('Restorative Care Places, unique non-numeric values:')
print(df_raw['Restorative Care Places'].value_counts().head(10).to_string())

Restorative Care Places, unique non-numeric values:
Restorative Care Places
      5185
10      23
15      12
20      11
9        7
5        6
8        6
25       6
16       5
12       5


### 3.10 Check Latitude and Longitude Range

In [18]:
# Confirm coordinates are within Australia's expected geographic bounds
# Australia: Latitude -44 to -10, Longitude 113 to 154
print(f"Latitude  — Min: {df_raw['Latitude'].min():.4f}  Max: {df_raw['Latitude'].max():.4f}")
print(f"Longitude — Min: {df_raw['Longitude'].min():.4f}  Max: {df_raw['Longitude'].max():.4f}")

# Flag any coordinates outside Australia's bounds
out_of_bounds = df_raw[
    (df_raw['Latitude'] < -44) | (df_raw['Latitude'] > -10) |
    (df_raw['Longitude'] < 113) | (df_raw['Longitude'] > 154)
]
print(f'\nRecords with coordinates outside Australia: {len(out_of_bounds)}')

Latitude  — Min: -43.3143  Max: -10.5698
Longitude — Min: 113.6610  Max: 167.9548

Records with coordinates outside Australia: 2


---
## 4. Data Cleaning

Based on the EDA above, we apply the following cleaning steps:

| Step | Action | Reason |
|------|--------|--------|
| 4.1 | Drop funding column | Not needed for location finder feature |
| 4.2 | Remove Innovative Pool records | Only 2 records, not a recognisable walk-in venue |
| 4.3 | Convert postcode to string | Prevent arithmetic, preserve formatting |
| 4.4 | Fix Restorative Care Places | Replace space strings with 0 |
| 4.5 | Fix Home Care Places null | Fill single null with 0 |
| 4.6 | Strip whitespace from strings | Remove leading/trailing spaces from text fields |

In [19]:
# Work on a copy so the raw dataframe stays intact for reference
df = df_raw.copy()

### 4.1 Drop Irrelevant Column

In [20]:
# Drop the government funding column — financial data not needed
# for the location finder feature
df = df.drop(columns=['2024-25 Australian Government Funding'])

print(f'Columns after drop: {df.shape[1]}')

Columns after drop: 24


### 4.2 Remove Innovative Pool Records

In [21]:
# Innovative Pool: only 2 records nationally, experimental funding
# arrangements — not a physical venue older Australians can visit
before = len(df)
df = df[df['Care Type'] != 'Innovative Pool'].copy()
after = len(df)

print(f'Removed {before - after} Innovative Pool record(s)')
print(f'Rows remaining: {after}')

Removed 2 Innovative Pool record(s)
Rows remaining: 5376


### 4.3 Fix Postcode Format

In [22]:
# Convert postcode from integer to string
# This prevents accidental arithmetic and keeps it usable for text search
df['Physical Post Code'] = df['Physical Post Code'].astype(str).str.strip()

print(f"Postcode dtype after fix : {df['Physical Post Code'].dtype}")
print(f"Sample postcodes         : {df['Physical Post Code'].head(5).tolist()}")

Postcode dtype after fix : object
Sample postcodes         : ['2622', '2640', '2839', '2477', '2645']


### 4.4 Fix 'Restorative Care Places' Replace Space Strings

In [24]:
# Replace ' ' (space string) with 0 — these rows simply have no
# restorative care places, the space is a data entry artifact
df['Restorative Care Places'] = (
    pd.to_numeric(df['Restorative Care Places'], errors='coerce')
    .fillna(0)
    .astype(int)
)

print("'Restorative Care Places' cleaned.")
print(f"Unique values sample: {sorted(df['Restorative Care Places'].unique())[:8]}")

'Restorative Care Places' cleaned.
Unique values sample: [0, 2, 3, 4, 5, 6, 7, 8]


### 4.5 Fix 'Home Care Places' Fill Single Null

In [25]:
# One Transition Care row has a null Home Care Places value
# This is expected — Transition Care services don't have home care places
# Filling with 0 keeps the column consistent and numeric
null_count = df['Home Care Places'].isnull().sum()
df['Home Care Places'] = (
    pd.to_numeric(df['Home Care Places'], errors='coerce')
    .fillna(0)
    .astype(int)
)

print(f"Filled {null_count} null value(s) in 'Home Care Places' with 0.")

Filled 1 null value(s) in 'Home Care Places' with 0.


### 4.6 Strip Whitespace from All String Columns

In [27]:
# Leading/trailing spaces in text fields can cause silent mismatches
# during search and filtering strip them all
string_cols = df.select_dtypes(include='object').columns
df[string_cols] = df[string_cols].apply(lambda col: col.str.strip())

print(f'Whitespace stripped from {len(string_cols)} string columns.')

Whitespace stripped from 15 string columns.


## 5. Post-Cleaning Validation

In [28]:
# Reset index after row removals
df = df.reset_index(drop=True)

print('=== Final Dataset Summary ===')
print(f'Shape          : {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Rows removed   : {df_raw.shape[0] - df.shape[0]}')
print(f'Columns removed: {df_raw.shape[1] - df.shape[1]}')

=== Final Dataset Summary ===
Shape          : 5376 rows x 24 columns
Rows removed   : 2
Columns removed: 1


In [30]:
# Confirm zero nulls in cleaned dataset
print('Null values in cleaned dataset:')
print('-' * 35)
null_check = df.isnull().sum()
if null_check.sum() == 0:
    print('No nulls found , dataset is clean.')
else:
    print(null_check[null_check > 0].to_string())

Null values in cleaned dataset:
-----------------------------------
No nulls found , dataset is clean.


In [31]:
# Final care type distribution after cleaning
print('Care Type Distribution (cleaned):')
print('-' * 45)
print(df['Care Type'].value_counts().to_string())

Care Type Distribution (cleaned):
---------------------------------------------
Care Type
Residential                                                         2590
Home Care                                                           2363
Multi-Purpose Service                                                183
Short-Term Restorative Care (STRC)                                   126
Transition Care                                                       67
National Aboriginal and Torres Strait Islander Aged Care Program      47


In [32]:
# Preview cleaned dataset
df.head()

,Service Name,Physical Address,Physical Suburb,Physical State,Physical Post Code,2018 Aged Care Planning Region (ACPR),Care Type,Residential Places,Home Care Places,Restorative Care Places,Provider Name,Organisation Type,ABS Remoteness,2019 MMM Code,2016 SA2 Code,2016 SA2 Name,2016 SA3 Code,2016 SA3 Name,2023 LGA Name,2023 LGA Code,2017 PHN Code,2017 PHN Name,Latitude,Longitude
0,Braidwood Multi-Purpose Service,77 Monkittee Street,BRAIDWOOD,NSW,2622,Southern Highlands,Multi-Purpose Service,37,2,0,Southern NSW Local Health District,State Government,Inner Regional Australia,5,11007,Braidwood,10102,Queanbeyan,Queanbeyan-Palerang,16490,PHN106,South Eastern NSW,-35.442764,149.805407
1,Dellacourt,42 NICHOLSON Place,WEST ALBURY,NSW,2640,Riverina/Murray,Residential,122,0,0,Lutheran Aged Care Albury,Charitable,Inner Regional Australia,2,11174,Albury - South,10901,Albury,Albury,10050,PHN205,Murray,-36.075401,146.890896
2,Brerrina Multi-Purpose Service,56 DOYLE Street,BRERRINA,NSW,2839,Orana Far West,Multi-Purpose Service,12,4,0,Western NSW Local Health District,State Government,Very Remote Australia,7,11092,Bourke - Brewarrina,10501,Bourke - Cobar - Coonamble,Brewarrina,11200,PHN107,Western NSW,-29.961591,146.864313
3,BaptistCare Maranoa Centre - Alstonville,15 The Avenue -,ALSTONVILLE,NSW,2477,Far North Coast,Residential,90,0,0,BaptistCare NSW & ACT,Charitable,Inner Regional Australia,3,11237,Ballina Region,11201,Richmond Valley - Coastal,Ballina,10250,PHN109,North Coast,-28.840536,153.436634
4,Urana Multi-Purpose Service,127-129 Princess Street,URANA,NSW,2645,Riverina/Murray,Multi-Purpose Service,19,0,0,Murrumbidgee Local Health District,State Government,Outer Regional Australia,5,11181,Corowa Region,10903,Upper Murray exc. Albury,Federation,12870,PHN110,Murrumbidgee,-35.325391,146.267697


---
## 6. Export Cleaned Dataset

In [35]:
# CLEAN_PATH was defined in the load cell above
# Save to CSV- UTF-8 encoding to handle special characters in place names
df.to_csv(CLEAN_PATH, index=False, encoding='utf-8')

print(f'Cleaned dataset saved to: {CLEAN_PATH}')
print(f'Final shape: {df.shape[0]} rows x {df.shape[1]} columns')

Cleaned dataset saved to: C:\Monash\SEM4\FIT-5120-main\FIT-5120-Main-project\src\data\clean_data\cleaned_AIHW_ServiceList.csv
Final shape: 5376 rows x 24 columns
